In [10]:
### Ch3_5_MNIST_Classifier_Practice

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns


## 데이터 변환 설정
transforms = transforms.Compose([
    transforms.ToTensor(),
    # MNIST 데이터셋 전체 픽셀에 대한 평균(Mean)과 표준편차(Standard Deviation)
    transforms.Normalize((0.1307,), (0.3871,))
])


## MNIST 데이터셋 불러오기
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transforms, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transforms, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=True)


## MLP 모델 정의
class MLP_Final(nn.Module):
  # Layer의 입력과 출력은 데이터셋 크기와 일치해야 한다.
  # MNIST 데이터셋: 28×28 손글씨 이미지 ↔ 레이블: 0~9까지 총 10개 레이블
  def __init__(self):
    super(MLP_Final, self).__init__()
    self.fc1 = nn.Linear(28*28,128) # 입력 28×28
    self.fc2 = nn.Linear(128,64)
    self.fc3 = nn.Linear(64,10) # 최종 레이블 10개

  def forward(self, x):

    # 첫번째 변수 '-1'의 역할 → 28×28 사이즈에 맞게 shape 자동 설정
    x = x.view(-1, 28*28)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x


## GPU/모델/손실/옵티마이저 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP_Final().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## 최종 학습
print("최종 모델 학습 중...")

for epoch in range(3): # 총 3회 에포크
  model.train()
  # enumerate() → 반복 가능한 객체의 인덱스 & 요소를 순차적으로 반환한다.
  for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device) # 데이터셋을 GPU에 싣기
    # 기본 흐름: 기울기 초기화 → 예측 → 손실 측정 → 역전파 → 가중치 갱신
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    if batch_idx % 300 == 0:
      print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

최종 모델 학습 중...
Epoch 1, Batch 0, Loss: 2.2831
Epoch 1, Batch 300, Loss: 0.4114
Epoch 1, Batch 600, Loss: 0.1873
Epoch 1, Batch 900, Loss: 0.1918
Epoch 2, Batch 0, Loss: 0.1935
Epoch 2, Batch 300, Loss: 0.0802
Epoch 2, Batch 600, Loss: 0.0830
Epoch 2, Batch 900, Loss: 0.0728
Epoch 3, Batch 0, Loss: 0.0934
Epoch 3, Batch 300, Loss: 0.0997
Epoch 3, Batch 600, Loss: 0.0398
Epoch 3, Batch 900, Loss: 0.0662
